# Solving the 1D Poisson Equation with libCEED and SciPy

We use libCEED and SciPy’s conjugate gradient solver to solve the 1D Poisson equation $-u''(x)=\pi^2\sin(\pi x)$ on $[0,1]$, with solution fixed at zero at both the ends $u(0)=u(1)=0$. Since we know the exact solution is $u(x)=\sin(\pi x)$, we can check our result by comparing the numerical and exact values at the mesh nodes.

In [1]:
from pathlib import Path
import ctypes

lib_path = Path.home() / "comp526/libCEED/lib/libceed.dylib"
print("Library exists:", lib_path.is_file())

if lib_path.is_file():
    ceed_library = ctypes.CDLL(str(lib_path))
    print("Library loaded")

Library exists: True
Library loaded


## Set up the 1D Poisson operator

We divide the interval $[0,1]$ into 15 elements. This step builds the libCEED operator that SciPy will use to solve the equation without assembling a matrix.

In [ ]:
import libceed
import numpy as np

ceed = libceed.Ceed()

num_elem = 15
p = 5
q = 8
num_x = num_elem + 1
num_u = num_elem*(p-1) + 1

# Vectors
x = ceed.Vector(num_x)
x_array = np.zeros(num_x)
for i in range(num_x):
  x_array[i] = i / (num_x - 1.0)
x.set_array(x_array, cmode=libceed.USE_POINTER)

q_data = ceed.Vector(num_elem*q)
u = ceed.Vector(num_u)
v = ceed.Vector(num_u)

# Restrictions
indices_x = np.zeros(num_elem * 2, dtype="int32")
for i in range(num_elem):
    indices_x[2*i] = i
    indices_x[2*i + 1] = i + 1
restriction_x = ceed.ElemRestriction(num_elem, 2, 1, 1, num_x, indices_x, cmode=libceed.USE_POINTER)

indices_u = np.zeros(num_elem*p, dtype="int32")
for i in range(num_elem):
  for j in range(p):
    indices_u[p*i+j] = i*(p-1) + j
restriction_u = ceed.ElemRestriction(num_elem, p, 1, 1, num_u, indices_u, cmode=libceed.USE_POINTER)
strides = np.array([1, q, q], dtype="int32")
restriction_q_data = ceed.StridedElemRestriction(num_elem, q, 1, q*num_elem, strides)

# Bases
basis_x = ceed.BasisTensorH1Lagrange(1, 1, 2, q, libceed.GAUSS)
basis_u = ceed.BasisTensorH1Lagrange(1, 1, p, q, libceed.GAUSS)

# QFunctions
qf_setup = ceed.QFunctionByName("Poisson1DBuild")
qf_mass = ceed.QFunctionByName("Poisson1DApply")

# Setup operator
op_setup = ceed.Operator(qf_setup)
op_setup.set_field("dx", restriction_x, basis_x, libceed.VECTOR_ACTIVE)
op_setup.set_field("weights", libceed.ELEMRESTRICTION_NONE, basis_x,
                   libceed.VECTOR_NONE)
op_setup.set_field("qdata", restriction_q_data, libceed.BASIS_NONE,
                   libceed.VECTOR_ACTIVE)
op_setup.check()
print('Setup operator: ', op_setup)

# Poisson operator
op_poisson = ceed.Operator(qf_mass)
op_poisson.set_field("du", restriction_u, basis_u, libceed.VECTOR_ACTIVE)
op_poisson.set_field("qdata", restriction_q_data, libceed.BASIS_NONE, q_data)
op_poisson.set_field("dv", restriction_u, basis_u, libceed.VECTOR_ACTIVE)
op_poisson.check()
print('Poisson operator: ', op_poisson)

# Setup
op_setup.apply(x, q_data)


Setup operator:  CeedOperator - Poisson1DBuild
  15 elements with 8 quadrature points each
  3 fields
  2 input fields:
    Input field 0:
      Name: "dx"
      Size: 1
      EvalMode: gradient
      Active vector
    Input field 1:
      Name: "weights"
      Size: 1
      EvalMode: quadrature weights
      No vector
  1 output field:
    Output field 0:
      Name: "qdata"
      Size: 1
      EvalMode: none
      No basis
      Active vector

Poisson operator:  CeedOperator - Poisson1DApply
  15 elements with 8 quadrature points each
  3 fields
  2 input fields:
    Input field 0:
      Name: "du"
      Size: 1
      EvalMode: gradient
      Active vector
    Input field 1:
      Name: "qdata"
      Size: 1
      EvalMode: none
      No basis
  1 output field:
    Output field 0:
      Name: "dv"
      Size: 1
      EvalMode: gradient
      Active vector



In [ ]:
from scipy.sparse.linalg import LinearOperator, cg

def apply_interior(trial):
    with u.array_write() as u_values:
        u_values[:] = 0
        u_values[1:-1] = np.asarray(trial).reshape(-1)

    op_poisson.apply(u, v)

    with v.array_read() as v_values:
        return v_values[1:-1].copy()

n = num_u - 2
A = LinearOperator((n, n), matvec=apply_interior, dtype=x_array.dtype)

print("Operator shape:", A.shape)

Operator shape: (59, 59)


In [18]:
points, _ = ceed.lobatto_quadrature(p)
x_nodes = np.zeros(num_u)

for elem in range(num_elem):
    start = elem * (p - 1)
    x_nodes[start:start + p] = (elem + (points + 1) / 2) / num_elem

f_nodes = np.pi**2 * np.sin(np.pi * x_nodes)

print("Number of solution nodes:", x_nodes.size)
print("First and last coordinates:", x_nodes[0], x_nodes[-1])
print("Forcing at the midpoint:", f_nodes[num_u // 2])

Number of solution nodes: 61
First and last coordinates: 0.0 1.0
Forcing at the midpoint: 9.869604401089358


In [19]:
# Compute the integration data for the mass operator on this mesh
mass_q_data = ceed.Vector(num_elem * q)
mass_setup = ceed.Operator(ceed.QFunctionByName("Mass1DBuild"))
mass_setup.set_field("dx", restriction_x, basis_x, libceed.VECTOR_ACTIVE)
mass_setup.set_field("weights", libceed.ELEMRESTRICTION_NONE, basis_x,
                     libceed.VECTOR_NONE)
mass_setup.set_field("qdata", restriction_q_data, libceed.BASIS_NONE,
                     libceed.VECTOR_ACTIVE)
mass_setup.apply(x, mass_q_data)

# Apply the mass operator to the forcing values
mass = ceed.Operator(ceed.QFunctionByName("MassApply"))
mass.set_field("u", restriction_u, basis_u, libceed.VECTOR_ACTIVE)
mass.set_field("qdata", restriction_q_data, libceed.BASIS_NONE, mass_q_data)
mass.set_field("v", restriction_u, basis_u, libceed.VECTOR_ACTIVE)

forcing = ceed.Vector(num_u)
load = ceed.Vector(num_u)
with forcing.array_write() as values:
    values[:] = f_nodes

mass.apply(forcing, load)

with load.array_read() as values:
    b = values[1:-1].copy()
    print("Full load sum:", np.sum(values))
    print("Interior RHS size:", b.size)

Full load sum: 6.2831853071796
Interior RHS size: 59


In [20]:
u_interior, info = cg(A, b, rtol=1e-10, atol=0.0)

u_solution = np.zeros(num_u)
u_solution[1:-1] = u_interior

u_exact = np.sin(np.pi * x_nodes)
print("CG status:", info)
print("Numerical solution at midpoint:", u_solution[num_u // 2])
print("Maximum error at the nodes:", np.max(np.abs(u_solution - u_exact)))

CG status: 0
Numerical solution at midpoint: 0.9999999999129252
Maximum error at the nodes: 8.707479182135103e-11
